In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
data = pd.read_csv(
    "../data/processed/final_processed_dataset.csv"
)

print(data.shape)

data.head()

(11010, 45)


,farmer_id,weather_condition,collection_shift,pH,pH_duplicate,turbidity_ntu,turbidity_log,temperature_c_x,ammonia_content,color_score,...,temp_max,temp_min,temp_mean,humidity_max,humidity_min,precipitation,wind_speed,storage_risk,temp_humidity_index,vfa
0,0.652174,0.666667,1.0,0.465690,0.465541,0.086036,0.337074,0.377073,0.219000,1.00,...,0.507725,0.492275,0.507725,0.507725,0.492275,0.492275,0.492275,0.0,0.511017,0.073
1,0.673913,0.000000,1.0,0.479828,0.480012,0.069748,0.273708,0.397927,0.234667,0.75,...,0.507725,0.492275,0.507725,0.507725,0.492275,0.492275,0.492275,0.0,0.511017,0.061
2,0.652174,1.000000,1.0,0.532500,0.532451,0.053599,0.190202,0.391341,0.243000,0.50,...,0.507725,0.492275,0.507725,0.507725,0.492275,0.492275,0.492275,0.0,0.511017,0.036
3,0.586957,0.333333,1.0,0.427845,0.428358,0.098205,0.375919,0.392561,0.207333,1.00,...,0.507725,0.492275,0.507725,0.507725,0.492275,0.492275,0.492275,0.0,0.511017,0.094
4,0.369565,1.000000,1.0,0.465862,0.465722,0.082577,0.324852,0.369756,0.223667,1.00,...,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.0,1.000000,0.071


In [3]:
X = data.drop(columns=[
    'vfa',
    'pH',
    'pH_duplicate',
    'turbidity_ntu',
    'turbidity_log',
    'ammonia_content',
    'drc',
    'color_score',
    'farmer_id'
], errors='ignore')

y = data['vfa']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [5]:
X_train = np.array(X_train)
X_test = np.array(X_test)

X_train = X_train.reshape(
    X_train.shape[0],
    1,
    X_train.shape[1]
)

X_test = X_test.reshape(
    X_test.shape[0],
    1,
    X_test.shape[1]
)

print(X_train.shape)
print(X_test.shape)

(8808, 1, 36)
(2202, 1, 36)


In [6]:
model = Sequential()

model.add(
    LSTM(
        64,
        activation='relu',
        input_shape=(X_train.shape[1], X_train.shape[2])
    )
)

model.add(Dropout(0.2))

model.add(Dense(32, activation='relu'))

model.add(Dense(1))

model.compile(
    optimizer='adam',
    loss='mse'
)

model.summary()

F:\Y4S1\RP\FARMER-APP\R26-IT-120\LSTM-ML\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Layer (type)                ┃ Output Shape        ┃     Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ lstm (LSTM)                 │ (None, 64)          │      25,856 │
├─────────────────────────────┼─────────────────────┼─────────────┤
│ dropout (Dropout)           │ (None, 64)          │           0 │
├─────────────────────────────┼─────────────────────┼─────────────┤
│ dense (Dense)               │ (None, 32)          │       2,080 │
├─────────────────────────────┼─────────────────────┼─────────────┤
│ dense_1 (Dense)             │ (None, 1)           │          33 │
└─────────────────────────────┴─────────────────────┴─────────────┘

 Total params: 27,969 (109.25 KB)

 Trainable params: 27,969 (109.25 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 9.4616e-04 - val_loss: 5.2341e-04
Epoch 2/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 5.2879e-04 - val_loss: 5.2466e-04
Epoch 3/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 5.2863e-04 - val_loss: 5.2395e-04
Epoch 4/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 5.2978e-04 - val_loss: 5.2335e-04
Epoch 5/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 5.2953e-04 - val_loss: 5.2338e-04
Epoch 6/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 5.2923e-04 - val_loss: 5.2528e-04
Epoch 7/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 5.2976e-04 - val_loss: 5.2344e-04
Epoch 8/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 5.2913e-04 - val_loss: 5.2708e-04
Epoch 9/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 5.3065e-04 - val_loss: 5.2397e-04
Epoch 10/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 5.3072e-04 - val_loss: 5.2412e-04
Epoch 11/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss

In [8]:
y_pred = model.predict(X_test)

y_pred = y_pred.flatten()

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("LSTM MAE:", mae)
print("LSTM RMSE:", rmse)
print("LSTM R2 Score:", r2)

69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
LSTM MAE: 0.020265538960126832
LSTM RMSE: 0.023002027391316158
LSTM R2 Score: -8.199499537120403e-06
